# Session 13 lab: regression, explaining and predicting a number

You will fit a simple and a multiple linear regression on the Lakeside Mart weekly data, read every coefficient as a sentence, check the residuals, meet the dummy variable trap, test the model on weeks it never saw, and save a summary file for your Project milestone 2 report.

In every step: **PREDICT**, run, **READ**, **CHANGE** one thing. Cells with `# your code here` are yours to fill in. Until you fill them in, the checks say NOT YET and the notebook keeps running. Cells with a `check(...)` line tell you whether your number matches.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)

# Helper: run a check on a value that may not exist yet. You do not need to read it.
def safe(f):
    try:
        return f()
    except Exception:
        return np.nan

---
## Step 1: Look at the data

Each record of branch_weeks.csv is one branch in one week. Footfall is the door count, promo_on is 1 in a promotion week, orders is the target.

**PREDICT first.** Will the correlation of orders with footfall be strongly positive, weakly positive, or near zero? Roughly what number?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
bw = pd.read_csv("branch_weeks.csv")
print(bw.shape)
print(bw.head(3))

r = np.nan  # your code here: the correlation of orders with footfall, using .corr()
print("correlation:", r)

plt.scatter(bw.footfall, bw.orders, s=6)
plt.xlabel("footfall"); plt.ylabel("orders"); plt.show()

check("832 records", len(bw), 832, 0)
check("correlation of orders with footfall", r, 0.8401, 0.01)

**READ the output.** Describe the cloud in one sentence. Is a straight line a fair summary?

> My answer: 

---
## Step 2: Simple linear regression by ordinary least squares

Solve for the intercept b0 and the coefficient b1 of footfall. X holds a column of ones (for the intercept) and the predictor.

**PREDICT first.** The slope will be small because footfall is in people and orders are counts. Will b1 be nearer 0.002, 0.02 or 0.2?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
X1 = np.column_stack([np.ones(len(bw)), bw.footfall])
y = bw.orders.values
simple = np.array([np.nan, np.nan])  # your code here: np.linalg.lstsq(X1, y, rcond=None)[0] gives b0 and b1
print(simple)

check("intercept b0", simple[0], 1.2291, 0.01)
check("coefficient b1", simple[1], 0.0185, 0.0005)

**READ the output.** Say b1 as a sentence for a manager: how many more orders go with 1,000 more people at the door?

> My answer: 

**CHANGE one thing.** Fit orders on rainfall_mm alone (replace bw.footfall in X1). Is the coefficient positive or negative? How does its size compare with footfall?

> What I saw: 

In [ ]:
Xr = np.column_stack([np.ones(len(bw)), bw.rainfall_mm])
rain_only = np.array([np.nan, np.nan])  # your code here: the same lstsq call with Xr
print(rain_only)

check("rain-only coefficient", rain_only[1], -0.1249, 0.005)

---
## Step 3: Multiple linear regression, residuals and R squared

Add promo_on and rainfall_mm. Then measure the fit: residuals are actual minus predicted, R squared is 1 minus the unexplained share of the variation.

**PREDICT first.** Will R squared rise above the 0.706 of footfall alone? By a lot or a little?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
cols = ["footfall", "promo_on", "rainfall_mm"]
X = np.column_stack([np.ones(len(bw)), bw[cols]])
coef = np.full(4, np.nan)  # your code here: lstsq on X and y, keep the first result
print(dict(zip(["intercept"] + cols, np.round(coef, 4))))

fitted = np.full(len(y), np.nan)  # your code here: X @ coef
residuals = np.full(len(y), np.nan)  # your code here: actual minus fitted
ss_res = np.nan  # your code here: sum of squared residuals
ss_tot = np.nan  # your code here: total variation of y around its mean
r2 = np.nan  # your code here: 1 minus ss_res over ss_tot
rmse = np.nan  # your code here: root of the mean squared residual
print("R squared:", round(r2, 3), " RMSE:", round(rmse, 2))

check("intercept", coef[0], 2.3785, 0.01)
check("footfall coefficient", coef[1], 0.0188, 0.0005)
check("promo_on coefficient", coef[2], 6.7986, 0.01)
check("rainfall_mm coefficient", coef[3], -0.1183, 0.001)
check("R squared", r2, 0.7946, 0.001)
check("RMSE", rmse, 6.0705, 0.01)

**READ the output.** Write one sentence for each of the three coefficients, starting "Holding the other predictors fixed ...".

> My answer: 

**CHANGE one thing.** Drop rainfall_mm from cols and refit. Does R squared fall a lot? What does that say about rainfall as a predictor?

> What I saw: 

---
## Step 4: Check the residuals, then add indicator variables

A residual plot should be an even cloud (homoscedasticity). Then turn the branch into indicator (dummy) variables with one level dropped.

**PREDICT first.** Will the residuals have the same spread for quiet and busy weeks, or will busy weeks spread more?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
plt.scatter(fitted, residuals, s=6)
plt.axhline(0, color="red")
plt.xlabel("predicted orders"); plt.ylabel("residual"); plt.show()

thirds = pd.qcut(pd.Series(fitted).rank(method="first"), 3, labels=False) if not np.isnan(fitted).any() else np.zeros(len(y), dtype=int)
sd_by_third = pd.Series([np.nan, np.nan, np.nan])  # your code here: pd.Series(residuals).groupby(thirds).std()
print(sd_by_third)

check("residual sd, lowest third", sd_by_third[0], 5.1664, 0.05)
check("residual sd, highest third", sd_by_third[2], 7.245, 0.05)

**READ the output.** Which pattern from the slide do you see: even cloud, curve, fan or outliers? What does it mean for forecasts of busy weeks?

> My answer: 

In [ ]:
dummies = pd.DataFrame(np.nan, index=bw.index, columns=["B02", "B03", "B04", "B05", "B06", "B07", "B08"])  # your code here: pd.get_dummies(bw.branch_id, drop_first=True, dtype=float)
Xb = np.full((len(bw), 11), np.nan)  # your code here: add the dummy columns to X with np.column_stack
cb = np.full(11, np.nan)  # your code here: lstsq on Xb and y
n, k = Xb.shape
r2b = np.nan  # your code here: R squared as before, using Xb and cb
adj_r2b = np.nan  # your code here: 1 - (1 - r2b) * (n - 1) / (n - k)
print(list(dummies.columns))
print("R squared", round(r2b, 3), "adjusted", round(adj_r2b, 3))

check("R squared with branch indicators", r2b, 0.7957, 0.001)
check("adjusted R squared with branch indicators", adj_r2b, 0.7932, 0.001)

**READ the output.** Adjusted R squared was 0.794 without the branch indicators. Did it rise or fall? Would you keep them?

> My answer: 

**CHANGE one thing.** Now keep all 8 indicators together with the intercept (drop_first=False) and print the matrix rank with np.linalg.matrix_rank. This is the dummy variable trap.

> What I saw: 

In [ ]:
full = np.column_stack([np.ones(len(bw)), pd.get_dummies(bw.branch_id, dtype=float)])
rank = np.nan  # your code here: np.linalg.matrix_rank(full)
print(full.shape, "columns, rank", rank)

check("matrix rank", rank, 8, 0)

---
## Step 5: Cross-check with the libraries

statsmodels and scikit-learn solve the same least squares problem. Their numbers should match your NumPy numbers.

**PREDICT first.** Will statsmodels give the same coefficients as NumPy, or slightly different ones?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

model = None  # your code here: smf.ols("orders ~ footfall + promo_on + rainfall_mm", data=bw).fit()   (the lines below wait until this is filled in)
if model is not None: print(model.params.round(4))
if model is not None: print("R squared:", round(model.rsquared, 3), " adjusted:", round(model.rsquared_adj, 3))
if model is not None: print(model.conf_int().round(3))

linear_model = None  # your code here: LinearRegression().fit(bw[cols], bw.orders)   (the line below waits until this is filled in)
if linear_model is not None: print(linear_model.intercept_, linear_model.coef_)

check("statsmodels matches NumPy", safe(lambda: model.params.values), coef, 0.0001)
check("statsmodels R squared", safe(lambda: model.rsquared), 0.7946, 0.001)
check("promo_on interval, lower end", safe(lambda: model.conf_int().loc["promo_on", 0]), 5.7928, 0.01)
check("scikit-learn matches NumPy", safe(lambda: np.r_[linear_model.intercept_, linear_model.coef_]), coef, 0.0001)

**READ the output.** statsmodels adds the intercept for you in a formula. What other things does model.summary() report that NumPy did not?

> My answer: 

---
## Step 6: Predict, and test on weeks the model never saw

To predict, fit on the first 78 weeks (the training set) and score on the last 26 weeks (the test set). The baseline model predicts the training mean.

**PREDICT first.** A branch expects 2,000 people, a promotion week and 20 mm of rain. About how many orders?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
week = np.array([1, 2000, 1, 20])      # the 1 is for the intercept
pred = np.nan  # your code here: week @ coef
print("predicted orders:", round(pred, 2))

check("predicted orders for the week", pred, 44.3484, 0.05)

In [ ]:
is_train = (bw.week_start < "2025-06-30").values
coef_tr = np.full(4, np.nan)  # your code here: lstsq on the training rows only
pred_te = X[~is_train] @ coef_tr
rmse_model = np.nan  # your code here: RMSE of the model on the test rows
rmse_baseline = np.nan  # your code here: RMSE if every test week gets the training mean
print(is_train.sum(), "training records,", (~is_train).sum(), "test records")
print("test RMSE: model", round(rmse_model, 2), " baseline", round(rmse_baseline, 2))

check("training records", is_train.sum(), 624, 0)
check("test RMSE of the model", rmse_model, 6.2932, 0.05)
check("test RMSE of the baseline model", rmse_baseline, 14.582, 0.05)

**READ the output.** Did the regression beat the baseline model on weeks it never saw? By how many orders?

> My answer: 

**CHANGE one thing.** Change the week to 2,000 people with no promotion (the third number 0). How many orders fewer?

> What I saw: 

---
## Step 7: Keep a summary file for your report

A coefficient table with 95 percent intervals is the heart of your one page report. The interval is the coefficient plus or minus 1.963 standard errors (Student t, 828 degrees of freedom).

In [ ]:
s2 = (residuals ** 2).sum() / (len(y) - X.shape[1])
se = np.full(4, np.nan)  # your code here: np.sqrt(s2 * np.diag(np.linalg.inv(X.T @ X)))
ci_low = np.full(4, np.nan)  # your code here: coef - 1.963 * se
ci_high = np.full(4, np.nan)  # your code here: coef + 1.963 * se
summary = pd.DataFrame({"term": ["intercept"] + cols, "coefficient": coef, "ci_low": ci_low, "ci_high": ci_high})
summary.to_csv("regression_summary.csv", index=False)
summary.to_excel("regression_summary.xlsx", index=False)   # opens in Excel or Power BI
print(summary.round(4))

check("promo_on interval, lower", summary.ci_low[2], 5.7928, 0.01)
check("promo_on interval, upper", summary.ci_high[2], 7.8044, 0.01)
check("rainfall interval, upper", summary.ci_high[3], -0.1004, 0.003)

## Your one page report (Project milestone 2)

Double-click this cell and write, in plain sentences:

1. **Question:** the target, the predictors and the decision.
2. **Model:** multiple linear regression, with the unit of every predictor.
3. **Findings:** three coefficients as sentences with their intervals, and R squared.
4. **Check:** what the residual plot showed and what you would do about it.
5. **Limits:** what the model cannot prove, and the test RMSE against the baseline model.
6. **Project question:** one sentence.

> My report: 

In [ ]:
show_checks()